In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
from certifiable_learning_stability.dpa_certifier import HalfmoonsCertifier, MnistCertifier, Cifar10Certifier
from certifiable_learning_stability.rdp_certifier import Cifar10Certifier as Cifar10RDPCertifier
from certifiable_learning_stability.certification_methods import CertificationMethod, AggregationType
from experiments.reproducibility import make_reproducible, get_device
from experiments.misc import dummy_hparams_agt, dummy_hparams_hrdp, dummy_hparams_prdp, dummy_hparams_sgd
import torch

In [ ]:
SEED = 42
make_reproducible(SEED)

# CIFAR-10

## Vanilla DPA - Bare Bones Resnet18

### Pointwise RDP

In [ ]:
hyperparams = {
    "epochs": 40,
    "lr": 0.001,
    "mechanism_samples": 250,
    "confidence": 0.98,
    "seed": SEED,
    "sigma": 0.3,
    "max_grad_norm": 25.0,
    "sample_rate": 128 / 10000,
    "sub_training_size": 10000,
}
device = get_device(index=0)

In [ ]:
kwargs = {"logfile_name": "baseline_cifar10", "write_to_file": True, "save": True}
cifar10_rdp_certifier = Cifar10RDPCertifier(hyperparams, device, save_kwargs=kwargs, pretrained=False)


In [ ]:
cert_dict = cifar10_rdp_certifier.certify_points("dp_bagging_softmax_prob")
cert_dict

### Vanilla DPA/ROE

In [ ]:
hp_hrdp_cifar = dummy_hparams_hrdp()
hp_agt_cifar = dummy_hparams_agt()
hp_prdp_cifar = dummy_hparams_prdp()
hp_sgd_cifar = {
    "epochs": 50,
    "batch_size": 512,
    "lr": 1e-5,
    "weight_decay": 1e-3,
}
hyperparams_dpa_cifar = {
    "num_partitions": 50,
    "test_batch_size": 500,
    "seed": SEED,
    "method_name": "dpa_vanilla_bare_bones_resnet",
    "hp_sgd": hp_sgd_cifar,
    "hp_agt": hp_agt_cifar,
    "hp_hrdp": hp_hrdp_cifar,
    "hp_prdp": hp_prdp_cifar,
}
device = get_device(index=1)

In [ ]:
kwargs = {"logfile_name": "generalized_framework", "write_to_file": True}
cifar10_dpa_certifier = Cifar10Certifier(hyperparams_dpa_cifar, device, save_kwargs=kwargs)

In [ ]:
cifar10_dpa_certifier.train_dpa_partitions((CertificationMethod.SGD,), partitioning_method="disjoint_bag")

In [ ]:
cifar10_dpa_certifier.get_metrics_inference(
    CertificationMethod.SGD,
    agg_type=AggregationType.DPA,
)
cifar10_dpa_certifier.get_metrics_inference(
    CertificationMethod.SGD,
    agg_type=AggregationType.ROE
)

### DPA/ROE + Pointwise RDP

In [ ]:
hp_hrdp_cifar = dummy_hparams_hrdp()
hp_agt_cifar = dummy_hparams_agt()
hp_sgd_cifar = dummy_hparams_sgd()
hp_prdp_cifar = {
    "epochs": 45,
    "lr": 0.001,
    "mechanism_samples": 2,
    "confidence": 0.95,
    "seed": SEED,
    "sigma": 0.35,
    "max_grad_norm": 22.0,
    "sample_rate": 128 / 5000,
    "sub_training_size": 5000,
}
hyperparams_dpa_cifar = {
    "num_partitions": 5,
    "test_batch_size": 100,
    "seed": SEED,
    "method_name": "dpa_prdp_bagging_raw_resnet",
    "hp_sgd": hp_sgd_cifar,
    "hp_agt": hp_agt_cifar,
    "hp_hrdp": hp_hrdp_cifar,
    "hp_prdp": hp_prdp_cifar,
}
device = get_device(index=0)

In [ ]:
kwargs = {"logfile_name": "generalized_framework", "write_to_file": True}
cifar10_dpa_certifier = Cifar10Certifier(hyperparams_dpa_cifar, device, save_kwargs=kwargs)

In [ ]:
cifar10_dpa_certifier.train_dpa_partitions((CertificationMethod.POINTWISE_RDP,), partitioning_method="disjoint_bag")

In [ ]:
cifar10_dpa_certifier.get_metrics_inference(
    CertificationMethod.POINTWISE_RDP,
    agg_type=AggregationType.DPA,
    # 1000 randomly sampled points from the test set
    test_set=cifar10_dpa_certifier.test_set[torch.randint(0, len(cifar10_dpa_certifier.test_set), (1000,))],
)
# cifar10_dpa_certifier.get_metrics_inference(
#     CertificationMethod.SGD,
#     agg_type=AggregationType.ROE
# )

## Vanilla DPA - Finetuned Resnet18 (Non-Frozen Resnet Block)

### Pointwise RDP

In [ ]:
hyperparams = {
    "epochs": 12,
    "lr": 0.001,
    "mechanism_samples": 250,
    "confidence": 0.98,
    "seed": SEED,
    "sigma": 0.4,
    "max_grad_norm": 22.0,
    "sample_rate": 128 / 10000,
    "weight_decay": 0.0001,
    "sub_training_size": 10000,
}
device = get_device(index=0)

In [ ]:
kwargs = {"logfile_name": "baseline_cifar10", "write_to_file": True, "load": True}
cifar10_rdp_certifier = Cifar10RDPCertifier(hyperparams, device, save_kwargs=kwargs, pretrained=True)


In [ ]:
cert_dict = cifar10_rdp_certifier.certify_points("dp_bagging_softmax_prob")
cert_dict

### Vanilla DPA/ROE

In [ ]:
hp_hrdp_cifar = dummy_hparams_hrdp()
hp_agt_cifar = dummy_hparams_agt()
hp_prdp_cifar = dummy_hparams_prdp()
hp_sgd_cifar = {
    "epochs": 25,
    "batch_size": 512,
    "lr": 2.5e-4,
    "weight_decay": 1e-3,
}
hyperparams_dpa_cifar = {
    "num_partitions": 50,
    "test_batch_size": 500,
    "seed": SEED,
    "method_name": "dpa_vanilla_fine_tuned_resnet",
    "hp_sgd": hp_sgd_cifar,
    "hp_agt": hp_agt_cifar,
    "hp_hrdp": hp_hrdp_cifar,
    "hp_prdp": hp_prdp_cifar,
}
device = get_device(index=1)

In [ ]:
kwargs = {"logfile_name": "generalized_framework", "write_to_file": True}
cifar10_dpa_certifier = Cifar10Certifier(hyperparams_dpa_cifar, device, save_kwargs=kwargs, pre_trained=True)

In [ ]:
cifar10_dpa_certifier.train_dpa_partitions((CertificationMethod.SGD,), partitioning_method="disjoint_bag")

In [ ]:
cifar10_dpa_certifier.get_metrics_inference(
    CertificationMethod.SGD,
    agg_type=AggregationType.DPA,
)
cifar10_dpa_certifier.get_metrics_inference(
    CertificationMethod.SGD,
    agg_type=AggregationType.ROE
)

### DPA/ROE + Pointwise RDP

In [ ]:
SEED = 42
make_reproducible(SEED)
hp_hrdp_cifar = dummy_hparams_hrdp()
hp_sgd_cifar = dummy_hparams_sgd()
hp_agt_cifar = dummy_hparams_agt()
hp_prdp_cifar = {
    "epochs": 15,
    "lr": 0.0025,
    "mechanism_samples": 2,
    "confidence": 0.98,
    "seed": SEED,
    "sigma": 0.4,
    "max_grad_norm": 22.0,
    "sample_rate": 100 / 3000,
    "sub_training_size": 3000,
    "weight_decay": 0.0005,
}
hyperparams_dpa_cifar = {
    "num_partitions": 5,
    "test_batch_size": 100,
    "seed": SEED,
    "method_name": "dpa_prdp_bagging_finetune_resnet",
    "hp_sgd": hp_sgd_cifar,
    "hp_agt": hp_agt_cifar,
    "hp_hrdp": hp_hrdp_cifar,
    "hp_prdp": hp_prdp_cifar,
}
device = get_device(index=0)

In [ ]:
kwargs = {"logfile_name": "generalized_framework", "write_to_file": True}
cifar10_ft_dpa_certifier = Cifar10Certifier(hyperparams_dpa_cifar, device, save_kwargs=kwargs, pre_trained=True)

In [ ]:
cifar10_ft_dpa_certifier.train_dpa_partitions((CertificationMethod.POINTWISE_RDP,), partitioning_method="disjoint_bag")

In [ ]:
# 1000 randomly sampled points from the test set
test_set_subset = torch.utils.data.Subset(cifar10_ft_dpa_certifier.test_set, indices=torch.randperm(len(cifar10_ft_dpa_certifier.test_set))[:1000])

cifar10_ft_dpa_certifier.get_metrics_inference(CertificationMethod.POINTWISE_RDP, agg_type=AggregationType.DPA, test_set=test_set_subset)
# cifar10_ft_dpa_certifier.get_metrics_inference(CertificationMethod.POINTWISE_RDP, agg_type=AggregationType.ROE, test_set=test_set_subset)

# HALFMOONS

In [ ]:

hp_agt_halfmoons = {
    "epochs": 4,
    "batch_size": 5000,
    "lr": 1,
    "lr_decay": 0.6,
    "lr_min": 1e-3,
    "weight_decay": 1e-4,
    "ks_private": list(range(1, 33, 4)),
    "clip_gammas": [0.1] # Seems to be optimal
}
hp_hrdp_halfmoons = dummy_hparams_hrdp()
hp_prdp_halfmoons = dummy_hparams_prdp()
hp_sgd_halfmoons = dummy_hparams_sgd()
hp_bag_halfmoons = {
    "num_partitions": 250,
    "test_batch_size": 500,
    "seed": SEED,
    "method_name": "bag_agt_halfmoons",
    "hp_sgd": hp_sgd_halfmoons,
    "hp_agt": hp_agt_halfmoons,
    "hp_hrdp": hp_hrdp_halfmoons,
    "hp_prdp": hp_prdp_halfmoons
}
device = get_device(index=1)

In [ ]:
kwargs = {"logfile_name": "generalized_framework", "write_to_file": True, "save": True}
halfmoons_dpa_certifier = HalfmoonsCertifier(hp_bag_halfmoons, device, save_kwargs=kwargs)

In [ ]:
halfmoons_dpa_certifier.train_dpa_partitions((CertificationMethod.AGT,), partitioning_method="bag")